# LSTM-ARO 重現：DJIA 30 檔原始資料集

重現 Gülmez (2023), *Stock price prediction with optimized deep LSTM network with artificial rabbits optimization algorithm*（ESWA 227, 120346）的六種模型比較：
**ANN、LSTM1D、LSTM2D、LSTM3D、LSTM-GA、LSTM-ARO**，另加不需訓練的 **NAIVE**（明天 = 今天）當基準。
設計說明見 `reproduction_plan.md`。

| | `USE_VALIDATION = False`（預設，照論文） | `USE_VALIDATION = True` |
|---|---|---|
| GA / ARO 的 fitness | **測試集** MSE（論文 Fig. 2 的作法） | 訓練集尾端 20% 驗證集 MSE |
| 訓練 | 固定 epochs，不做 early stopping | early stopping（搜尋 patience 5、最終 patience 10） |
| 注意 | 用測試集挑超參數 → GA/ARO 的測試結果偏樂觀 | 測試集只在最後用一次 |

## 協作者請看這裡（幫忙跑的人）

1. 開啟：[在 Colab 開啟這份 notebook](https://colab.research.google.com/github/F115118101/stock-price-prediction-via-LSTM-ARO/blob/main/implementation/djia_reproduction.ipynb)，然後「檔案 → 在雲端硬碟中儲存複本」。
2. 執行階段類型：**CPU 或 T4 GPU 都可以**。先用 T4 跑一次「時間估算」；如果顯示 CPU 比較快，就改成 CPU 執行階段，不必消耗 GPU 額度。
3. 在下一節「實驗設定」**只改 `SHARD`**（負責人會告訴你，例如 `"2/4"` = 4 份中的第 2 份），其他設定不要動，否則結果無法合併。
4. 「執行階段 → 全部執行」。第一次會要求連接 Google Drive，結果都存在你的 `MyDrive/lstm_aro_djia/`。
5. 分頁保持開著，大約每小時回來看一次（免費版會跳出「你還在嗎？」的確認，沒按會斷線）。
6. **斷線了**：重新連線後再按一次「全部執行」，會從中斷處接著跑（已完成的跳過，搜尋中已訓練過的候選直接讀紀錄）。最多損失正在訓練的那一個模型。
7. 全部跑完後，「匯出結果」那格會在 Drive 的 `lstm_aro_djia/exports/` 產生 zip 並下載，把 zip 傳給負責人。
   沒跑完也可以單獨執行那格，先交已完成的部分。

**負責人**：把收到的 zip 放到自己 Drive 的 `lstm_aro_djia/incoming/`，執行「合併」那格；設定或資料不一致的結果會被拒收。

## 1. 實驗設定

In [ ]:
# ===== 協作者只改這一行 =====
SHARD          = None      # 例如 "1/4"、"2/4"…；None = 自己跑全部股票

# ===== 以下設定所有人必須一致（改了就無法合併）=====
USE_VALIDATION = False     # False = 照論文（fitness 用 test、無 early stopping）；True = 切驗證集
PRICE_COL      = "Close"   # "Close"（只做分割調整，可與論文對照）或 "Adj Close"
POP_SIZE       = 10        # GA 與 ARO 共用（論文 50 × 50，太久）
ITERATIONS     = 10
SEARCH_EPOCHS  = 20        # 搜尋時每個候選模型的訓練 epochs
FINAL_EPOCHS   = 50        # 最終模型（含 ANN/LSTM1D/2D/3D）的訓練 epochs
FINAL_SEEDS    = 5         # 最終模型用幾個 seed 重訓取平均
BATCH_SIZE     = 32
SEED           = 42
DATA_URL       = "https://raw.githubusercontent.com/F115118101/stock-price-prediction-via-LSTM-ARO/main/implementation/data/djia30_2018_2022.csv"

# ===== 各自的執行環境（不影響結果）=====
WORKERS        = None      # None = 自動（Colab 2 vCPU → 2；16 執行緒的筆電 → 8）
USE_GPU        = None      # None = 由時間估算自動選較快的
SAVE_TO_DRIVE  = True      # Colab 上把資料與結果存到 Google Drive（斷線可續跑）
REBENCH        = False     # True = 重新做時間估算（否則沿用上次的結果）
TIME_BUDGET_H  = 5.0

## 2. 環境

In [ ]:
import os, sys, importlib, json, time, glob

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/lstm_aro_djia"
else:
    BASE = os.path.abspath(".")
os.makedirs(BASE, exist_ok=True)

try:
    import yfinance  # noqa: F401
except ImportError:
    !pip -q install yfinance

CPU = os.cpu_count() or 2
if WORKERS is None:
    WORKERS = min(8, CPU) if CPU <= 4 else min(8, CPU // 2)
THREADS = max(1, min(2, CPU // WORKERS))
print(f"Colab={IN_COLAB}  BASE={BASE}  CPU={CPU}  WORKERS={WORKERS}  threads/worker={THREADS}")

## 3. 程式庫

寫成 `djia_lib.py`，讓平行的 worker process 可以 import（Colab 與 Windows Jupyter 都用 spawn 啟動）。

In [ ]:
%%writefile djia_lib.py
"""djia_lib.py：Gülmez (2023) LSTM-ARO 在 DJIA 30 檔的重現。

由 djia_reproduction.ipynb 的 %%writefile 產生；要修改請改 notebook 那一格。
TensorFlow 只在 worker process 內延遲載入，讓每個 worker 能各自設定 thread 數與是否使用 GPU。
"""
import json
import math
import multiprocessing as mp
import os
import random
import time
import zlib
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd

# 論文 Table 7 的 30 檔
DJIA30 = ["AXP", "AMGN", "AAPL", "BA", "CAT", "CSCO", "CVX", "GS", "HD", "HON",
          "IBM", "INTC", "JNJ", "KO", "JPM", "MCD", "MMM", "MRK", "MSFT", "NKE",
          "PG", "TRV", "UNH", "CRM", "VZ", "V", "WBA", "WMT", "DIS", "DOW"]
BASELINES = ["NAIVE", "ANN", "LSTM1D", "LSTM2D", "LSTM3D"]
SEARCHERS = ["LSTM-GA", "LSTM-ARO"]
ALL_MODELS = BASELINES + SEARCHERS

# 論文 Table 6
OPTIMIZERS = ["Adagrad", "Adam", "Adamax", "RMSprop", "SGD"]
NEURONS = list(range(1, 21))
SEARCH_SPACE = [
    ("x0_lstm1_units", NEURONS),
    ("x1_lstm2_exists", [0, 1]),
    ("x2_lstm2_units", NEURONS),
    ("x3_lstm3_exists", [0, 1]),
    ("x4_lstm3_units", NEURONS),
    ("x5_dense_units", NEURONS),
    ("x6_dropout", [0.3, 0.4, 0.5, 0.6, 0.7]),
    ("x7_optimizer", OPTIMIZERS),
    ("x8_lr", [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]),
]
DIM = len(SEARCH_SPACE)

# 時間估算用：搜尋中實際需要訓練的比例（其餘命中快取），以及各 baseline 相對 LSTM2D 的訓練時間
UNIQUE_RATE = {"LSTM-ARO": 0.87, "LSTM-GA": 0.55}
BASELINE_COST = {"ANN": 0.35, "LSTM1D": 0.6, "LSTM2D": 1.0, "LSTM3D": 1.4}


@dataclass
class Config:
    data_csv: str = "data/djia30_2018_2022.csv"
    out_dir: str = "results_djia"
    start: str = "2018-01-01"
    end: str = "2023-01-01"
    price_col: str = "Close"          # "Close"（分割調整）或 "Adj Close"
    window: int = 20                  # 論文：前 20 天預測隔天
    train_ratio: float = 0.8          # 論文：80% train / 20% test
    # False = 照論文：fitness 用 test 誤差，訓練不做 early stopping（固定 epochs）
    # True  = 從 train 尾端切出驗證集算 fitness，並啟用 early stopping
    use_validation: bool = False
    val_ratio: float = 0.2
    search_patience: int = 5
    final_patience: int = 10
    pop_size: int = 10
    iterations: int = 10
    search_epochs: int = 20
    final_epochs: int = 50
    batch_size: int = 32
    final_seeds: int = 5
    seed: int = 42
    threads: int = 2
    use_gpu: bool = False
    data_hash: str = ""               # 資料檔 md5 前 8 碼；合併多人結果時用來確認大家用同一份資料


# 這些設定必須所有人一致，結果才能合併（threads、use_gpu、路徑不影響結果）
FP_FIELDS = ["price_col", "window", "train_ratio", "use_validation", "val_ratio", "search_patience",
             "final_patience", "search_epochs", "final_epochs", "batch_size", "final_seeds", "seed", "data_hash"]


def file_hash(path: str) -> str:
    import hashlib
    with open(path, "rb") as f:
        data = f.read().replace(bytes([13, 10]), bytes([10]))  # CRLF → LF，不受作業系統/git 換行設定影響
    return hashlib.md5(data).hexdigest()[:8]


def run_id(cfg: Config) -> str:
    """給協作者對照用：同一個 run_id 的結果才能合併。"""
    return f"{os.path.basename(run_dir(cfg))}_p{cfg.pop_size}_i{cfg.iterations}_data-{cfg.data_hash or 'none'}"


def shard_tickers(tickers: List[str], shard: Optional[str]) -> List[str]:
    """shard = "k/n"：輪流分配（第 k 份取 tickers[k-1::n]），各份的股票數與長短都較平均。"""
    if not shard:
        return list(tickers)
    k, n = (int(v) for v in shard.split("/"))
    if not 1 <= k <= n:
        raise ValueError(f"SHARD 應為 k/n 且 1 ≤ k ≤ n，收到 {shard!r}")
    return list(tickers)[k - 1::n]


def compatible(cfg: Config, info: dict) -> List[str]:
    """回傳與目前設定不一致的欄位（空 list = 可合併）。"""
    c = info.get("config", {})
    keys = list(FP_FIELDS) + (["pop_size", "iterations"] if info.get("model") in SEARCHERS else [])
    return [k for k in keys if c.get(k) != getattr(cfg, k)]


# ---------------------------------------------------------------- 路徑
def run_dir(cfg: Config) -> str:
    mode = "val" if cfg.use_validation else "paper"
    return os.path.join(cfg.out_dir, f"{mode}_e{cfg.search_epochs}-{cfg.final_epochs}_s{cfg.final_seeds}")


def model_dir(cfg: Config, ticker: str, model: str) -> str:
    name = f"{model}_p{cfg.pop_size}_i{cfg.iterations}" if model in SEARCHERS else model
    return os.path.join(run_dir(cfg), ticker, name)


# ---------------------------------------------------------------- TensorFlow（延遲載入）
_TF = None


def _tf(cfg: Config):
    global _TF
    if _TF is None:
        os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
        if not cfg.use_gpu:
            os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
        import tensorflow as tf
        try:
            tf.config.threading.set_intra_op_parallelism_threads(cfg.threads)
            tf.config.threading.set_inter_op_parallelism_threads(cfg.threads)
        except RuntimeError:
            pass
        gpus = tf.config.list_physical_devices("GPU")
        try:
            if cfg.use_gpu:
                for g in gpus:
                    tf.config.experimental.set_memory_growth(g, True)
            else:
                tf.config.set_visible_devices([], "GPU")
        except RuntimeError:
            pass
        _TF = tf
    return _TF


def set_seed(cfg: Config, seed: int):
    random.seed(seed)
    np.random.seed(seed)
    _tf(cfg).keras.utils.set_random_seed(seed)


# ---------------------------------------------------------------- 資料
def _flatten(df: pd.DataFrame) -> pd.DataFrame:
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def download_djia(cfg: Config, tickers: List[str] = DJIA30, extra_dir: Optional[str] = None):
    """下載一次並存成長格式 CSV。extra_dir/<TICKER>.csv（需有 Date、Close 欄）會優先使用，可用來補 WBA。"""
    import logging
    import warnings
    import yfinance as yf

    warnings.filterwarnings("ignore", category=FutureWarning)
    logging.getLogger("yfinance").setLevel(logging.CRITICAL)
    frames, missing = [], []
    for t in tickers:
        df = None
        p = os.path.join(extra_dir, f"{t}.csv") if extra_dir else None
        if p and os.path.exists(p):
            df = pd.read_csv(p)
            df = df.rename(columns={c: "Date" for c in df.columns if str(c).lower() in ("date", "datetime")})
        else:
            try:
                df = _flatten(yf.download(t, start=cfg.start, end=cfg.end, auto_adjust=False,
                                          progress=False)).reset_index()
            except Exception as e:  # noqa: BLE001
                print(f"  {t}: 下載失敗 {e}")
        if df is None or df.empty or "Close" not in df.columns:
            missing.append(t)
            continue
        date = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None).dt.normalize()
        frames.append(pd.DataFrame({
            "date": date, "ticker": t,
            "close": df["Close"].astype(float).values,
            "adj_close": df.get("Adj Close", df["Close"]).astype(float).values,
        }))
    out = pd.concat(frames, ignore_index=True)
    out = out[(out.date >= cfg.start) & (out.date < cfg.end)].dropna()
    os.makedirs(os.path.dirname(cfg.data_csv) or ".", exist_ok=True)
    out.to_csv(cfg.data_csv, index=False, lineterminator="\n")
    with open(cfg.data_csv + ".meta.json", "w", encoding="utf-8") as f:
        json.dump({"downloaded_at": time.strftime("%Y-%m-%d %H:%M:%S"), "start": cfg.start,
                   "end": cfg.end, "missing": missing}, f, ensure_ascii=False, indent=2)
    return out, missing


_PRICES: Dict[str, pd.DataFrame] = {}


def _price_table(cfg: Config) -> pd.DataFrame:
    if cfg.data_csv not in _PRICES:
        _PRICES[cfg.data_csv] = pd.read_csv(cfg.data_csv, parse_dates=["date"])
    return _PRICES[cfg.data_csv]


def available_tickers(cfg: Config) -> List[str]:
    have = set(_price_table(cfg).ticker)
    return [t for t in DJIA30 if t in have]


def load_prices(cfg: Config, ticker: str) -> pd.Series:
    df = _price_table(cfg)
    col = "adj_close" if cfg.price_col.lower().startswith("adj") else "close"
    s = df[df.ticker == ticker].set_index("date")[col].sort_index().astype(float).dropna()
    if s.empty:
        raise ValueError(f"{ticker} 沒有資料")
    return s


@dataclass
class Data:
    X_train: np.ndarray
    y_train: np.ndarray
    X_test: np.ndarray
    y_test: np.ndarray
    X_fit: np.ndarray          # 搜尋時訓練候選模型
    y_fit: np.ndarray
    X_eval: np.ndarray         # 搜尋時算 fitness（test 或 val）
    y_eval: np.ndarray
    lo: float
    hi: float
    test_dates: pd.DatetimeIndex
    prev_test: np.ndarray      # 測試集每一筆的前一天價格（NAIVE 預測值）

    def inverse(self, v) -> np.ndarray:
        return np.asarray(v, dtype=float).ravel() * (self.hi - self.lo) + self.lo


def prepare(prices: pd.Series, cfg: Config) -> Data:
    raw = prices.values.astype(float)
    n = len(raw) - cfg.window
    n_train = int(n * cfg.train_ratio)
    lo, hi = raw[: n_train + cfg.window].min(), raw[: n_train + cfg.window].max()  # 只看訓練段
    z = (raw - lo) / (hi - lo)
    X = np.lib.stride_tricks.sliding_window_view(z[:-1], cfg.window)[..., None].copy()
    y = z[cfg.window:].copy()
    X_train, y_train, X_test, y_test = X[:n_train], y[:n_train], X[n_train:], y[n_train:]
    if cfg.use_validation:
        n_val = int(n_train * cfg.val_ratio)
        X_fit, y_fit = X_train[:-n_val], y_train[:-n_val]
        X_eval, y_eval = X_train[-n_val:], y_train[-n_val:]
    else:
        X_fit, y_fit, X_eval, y_eval = X_train, y_train, X_test, y_test
    prev = raw[cfg.window + n_train - 1: -1]
    return Data(X_train, y_train, X_test, y_test, X_fit, y_fit, X_eval, y_eval, lo, hi,
                prices.index[cfg.window + n_train:], prev)


# ---------------------------------------------------------------- 模型（論文 Table 1–5）
def build_lstm(cfg: Config, lstm_units, dense_units, dropout, optimizer="Adam", lr=1e-3):
    k = _tf(cfg).keras
    m = k.Sequential([k.Input(shape=(cfg.window, 1))])
    for i, u in enumerate(lstm_units):
        m.add(k.layers.LSTM(u, return_sequences=i < len(lstm_units) - 1))
    m.add(k.layers.Dense(dense_units, activation="relu"))
    m.add(k.layers.Dropout(dropout))
    m.add(k.layers.Dense(1))
    m.compile(optimizer=getattr(k.optimizers, optimizer)(learning_rate=lr), loss="mse")
    return m


def build_baseline(cfg: Config, name: str):
    if name == "ANN":
        k = _tf(cfg).keras
        m = k.Sequential([
            k.Input(shape=(cfg.window, 1)), k.layers.Flatten(),
            k.layers.Dense(10, activation="relu"), k.layers.Dense(10, activation="relu"),
            k.layers.Dropout(0.5), k.layers.Dense(1),
        ])
        m.compile(optimizer=k.optimizers.Adam(learning_rate=1e-3), loss="mse")
        return m
    depth = {"LSTM1D": 1, "LSTM2D": 2, "LSTM3D": 3}[name]
    return build_lstm(cfg, [10] * depth, 10, 0.5)


def fit_model(cfg: Config, model, X, y, epochs, patience=None, val=None, val_split=None) -> int:
    """patience=None 時固定跑滿 epochs（論文模式）；否則依 val 或 val_split 做 early stopping。"""
    k = _tf(cfg).keras
    kw, cb = {}, []
    if patience:
        cb = [k.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)]
        if val is not None:
            kw["validation_data"] = val
        else:
            kw["validation_split"] = val_split
    h = model.fit(X, y, epochs=epochs, batch_size=cfg.batch_size, shuffle=True, verbose=0,
                  callbacks=cb, **kw)
    return len(h.history["loss"])


def predict(cfg: Config, model, X) -> np.ndarray:
    return model.predict(X, batch_size=256, verbose=0).ravel()


def metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """論文 Eq. 21–24，原始價格尺度；MAPE 回報比例（與論文表格數值一致）。"""
    e = y_true - y_pred
    mse = float(np.mean(e ** 2))
    return dict(MSE=mse, MAE=float(np.mean(np.abs(e))),
                MAPE=float(np.mean(np.abs(e / y_true))),
                R2=float(1 - np.sum(e ** 2) / np.sum((y_true - y_true.mean()) ** 2)))


# ---------------------------------------------------------------- 超參數編碼（論文 Table 6）
def decode(x: np.ndarray) -> dict:
    return {name: ch[min(max(int(float(v) * len(ch)), 0), len(ch) - 1)]
            for v, (name, ch) in zip(x, SEARCH_SPACE)}


def hp_to_arch(hp: dict) -> dict:
    units = [hp["x0_lstm1_units"]]
    if hp["x1_lstm2_exists"]:
        units.append(hp["x2_lstm2_units"])
    if hp["x3_lstm3_exists"]:
        units.append(hp["x4_lstm3_units"])
    return dict(lstm_units=units, dense_units=hp["x5_dense_units"], dropout=hp["x6_dropout"],
                optimizer=hp["x7_optimizer"], lr=hp["x8_lr"])


def arch_key(arch: dict) -> str:
    return json.dumps([arch["lstm_units"], arch["dense_units"], arch["dropout"], arch["optimizer"], arch["lr"]])


class Fitness:
    """fitness = 候選模型在 eval 集（原價格尺度）的 MSE。
    相同架構只訓練一次；每次評估寫進 candidates.jsonl，中斷後重跑會讀回來當快取。
    每個架構的訓練 seed 由架構本身決定，因此結果與評估順序無關。"""

    def __init__(self, cfg: Config, data: Data, log_path: str, base_seed: int):
        self.cfg, self.data, self.log_path, self.base_seed = cfg, data, log_path, base_seed
        self.cache: Dict[str, float] = {}
        self.n_calls = 0
        self.n_new = 0
        if os.path.exists(log_path):
            with open(log_path, encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        r = json.loads(line)
                        self.cache[r["key"]] = r["fitness"]

    def __call__(self, x: np.ndarray) -> float:
        self.n_calls += 1
        arch = hp_to_arch(decode(x))
        key = arch_key(arch)
        if key in self.cache:
            return self.cache[key]
        cfg, d = self.cfg, self.data
        _tf(cfg).keras.backend.clear_session()
        set_seed(cfg, self.base_seed + zlib.crc32(key.encode()) % 100_000)
        model = build_lstm(cfg, **arch)
        t = time.time()
        if cfg.use_validation:
            ep = fit_model(cfg, model, d.X_fit, d.y_fit, cfg.search_epochs,
                           patience=cfg.search_patience, val=(d.X_eval, d.y_eval))
        else:
            ep = fit_model(cfg, model, d.X_fit, d.y_fit, cfg.search_epochs)
        mse = float(np.mean((d.inverse(d.y_eval) - d.inverse(predict(cfg, model, d.X_eval))) ** 2))
        if not np.isfinite(mse):
            mse = float("inf")
        self.cache[key] = mse
        self.n_new += 1
        with open(self.log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(dict(key=key, arch=arch, fitness=mse, epochs=ep,
                                    seconds=round(time.time() - t, 2))) + "\n")
        return mse


# ---------------------------------------------------------------- ARO（論文 Eq. 6–20）
def aro(fobj, X0: np.ndarray, max_iter: int, rng: np.random.Generator, lb=0.0, ub=1.0):
    X = X0.copy()
    n, d = X.shape
    T = max_iter
    fit = np.array([fobj(x) for x in X])
    bi = int(np.argmin(fit))
    best_x, best_f = X[bi].copy(), float(fit[bi])
    history = [best_f]
    for t in range(1, T + 1):
        for i in range(n):
            L = (math.e - math.exp(((t - 1) / T) ** 2)) * math.sin(2 * math.pi * rng.random())  # Eq. 8
            c = np.zeros(d)
            c[rng.permutation(d)[: math.ceil(rng.random() * d)]] = 1                             # Eq. 9–10
            R = L * c                                                                            # Eq. 7
            A = 4 * (1 - t / T) * math.log(1 / max(rng.random(), 1e-12))                        # Eq. 20
            if A > 1:   # 迂迴覓食（探索）Eq. 6
                j = rng.choice([k for k in range(n) if k != i])
                v = X[j] + R * (X[i] - X[j]) + round(0.5 * (0.05 + rng.random())) * rng.standard_normal()
            else:       # 隨機躲藏（開發）Eq. 13, 16–18
                H = (T - t + 1) / T * rng.standard_normal()
                g = np.zeros(d)
                g[math.ceil(rng.random() * d) - 1] = 1
                b = X[i] + H * g * X[i]
                v = X[i] + R * (rng.random() * b - X[i])
            v = np.clip(v, lb, ub)
            fv = fobj(v)
            if fv < fit[i]:  # Eq. 19
                X[i], fit[i] = v, fv
                if fv < best_f:
                    best_x, best_f = v.copy(), float(fv)
        history.append(best_f)
    return best_x, best_f, history


# ---------------------------------------------------------------- GA（論文未給細節，採常見設定）
def ga(fobj, X0: np.ndarray, max_iter: int, rng: np.random.Generator, lb=0.0, ub=1.0,
       pc=0.8, pm=None, tournament=3):
    P = X0.copy()
    n, d = P.shape
    pm = 1.0 / d if pm is None else pm
    fit = np.array([fobj(x) for x in P])
    history = [float(fit.min())]

    def select():
        idx = rng.choice(n, size=min(tournament, n), replace=False)
        return P[idx[np.argmin(fit[idx])]]

    for _ in range(max_iter):
        children = [P[np.argmin(fit)].copy()]  # 菁英保留 1
        while len(children) < n:
            p1, p2 = select(), select()
            child = np.where(rng.random(d) < 0.5, p1, p2) if rng.random() < pc else p1.copy()
            mask = rng.random(d) < pm
            child[mask] = rng.uniform(lb, ub, mask.sum())
            children.append(child)
        P = np.array(children)
        fit = np.array([fobj(x) for x in P])
        history.append(float(fit.min()))
    b = int(np.argmin(fit))
    return P[b], float(fit[b]), history


# ---------------------------------------------------------------- 最終訓練（全部模型共用）
def final_eval(cfg: Config, data: Data, build_fn):
    rows, preds = [], []
    y_true = data.inverse(data.y_test)
    for s in range(cfg.final_seeds):
        _tf(cfg).keras.backend.clear_session()
        set_seed(cfg, cfg.seed + s)
        model = build_fn()
        t = time.time()
        if cfg.use_validation:
            ep = fit_model(cfg, model, data.X_train, data.y_train, cfg.final_epochs,
                           patience=cfg.final_patience, val_split=0.1)
        else:
            ep = fit_model(cfg, model, data.X_train, data.y_train, cfg.final_epochs)
        p = data.inverse(predict(cfg, model, data.X_test))
        rows.append(dict(seed=cfg.seed + s, **metrics(y_true, p), epochs=ep,
                         train_s=round(time.time() - t, 2)))
        preds.append(p)
    return rows, np.mean(preds, axis=0)


def search_seed(cfg: Config, ticker: str) -> int:
    return cfg.seed * 1000 + zlib.crc32(ticker.encode()) % 1000


def run_job(cfg_dict: dict, ticker: str, model: str) -> dict:
    """跑一檔股票的一種模型；已完成（有 done.json）就直接讀回結果。"""
    cfg = Config(**cfg_dict)
    out = model_dir(cfg, ticker, model)
    done = os.path.join(out, "done.json")
    if os.path.exists(done):
        with open(done, encoding="utf-8") as f:
            info = json.load(f)
        if not compatible(cfg, info):
            return info
        # 設定或資料不同：舊結果移到一旁，重新跑
        os.rename(out, f"{out}.stale-{time.strftime('%Y%m%d%H%M%S')}")
    os.makedirs(out, exist_ok=True)
    data = prepare(load_prices(cfg, ticker), cfg)
    y_true = data.inverse(data.y_test)
    info = dict(ticker=ticker, model=model, n_train=len(data.X_train), n_test=len(data.X_test))
    t0 = time.time()

    if model == "NAIVE":
        pred = data.prev_test
        rows = [dict(seed=0, **metrics(y_true, pred), epochs=0, train_s=0.0)]
    elif model in BASELINES:
        rows, pred = final_eval(cfg, data, lambda: build_baseline(cfg, model))
    elif model in SEARCHERS:
        ss = search_seed(cfg, ticker)
        X0 = np.random.default_rng(ss).uniform(0, 1, (cfg.pop_size, DIM))  # GA 與 ARO 共用初始族群
        fobj = Fitness(cfg, data, os.path.join(out, "candidates.jsonl"), ss)
        algo, rng = (aro, np.random.default_rng(ss + 1)) if model == "LSTM-ARO" else (ga, np.random.default_rng(ss + 2))
        ts = time.time()
        best_x, best_f, history = algo(fobj, X0, cfg.iterations, rng)
        search_s = time.time() - ts
        arch = hp_to_arch(decode(best_x))
        rows, pred = final_eval(cfg, data, lambda: build_lstm(cfg, **arch))
        info.update(best_arch=arch, best_fitness=best_f, history=history, n_calls=fobj.n_calls,
                    n_trained=len(fobj.cache), search_s=round(search_s, 1))
    else:
        raise ValueError(model)

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out, "metrics_seeds.csv"), index=False)
    pd.DataFrame({"date": data.test_dates, "actual": y_true, "pred": pred}).to_csv(
        os.path.join(out, "predictions.csv"), index=False)
    mean = df[["MSE", "MAE", "MAPE", "R2"]].mean()
    info.update({k: float(v) for k, v in mean.items()})
    info.update({f"{k}_std": float(df[k].std(ddof=0)) for k in ["MSE", "MAE", "MAPE", "R2"]})
    info.update(final_s_per_seed=float(df.train_s.mean()), total_s=round(time.time() - t0, 1),
                config=asdict(cfg))
    with open(done, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=1)
    return info


# ---------------------------------------------------------------- 平行執行
def _fmt(sec: float) -> str:
    return f"{int(sec // 3600)}h{int(sec % 3600 // 60):02d}m"


def run_jobs(cfg: Config, models: List[str], tickers: Optional[List[str]] = None, workers: int = 2) -> pd.DataFrame:
    """以 (股票, 模型) 為單位平行執行；用 spawn 啟動 worker，Colab 與 Windows Jupyter 都適用。"""
    tickers = tickers or available_tickers(cfg)
    jobs = [(t, m) for m in models for t in tickers]
    results, todo = [], []
    for t, m in jobs:
        p = os.path.join(model_dir(cfg, t, m), "done.json")
        info = None
        if os.path.exists(p):
            with open(p, encoding="utf-8") as f:
                info = json.load(f)
            bad = compatible(cfg, info)
            if bad:
                print(f"  {t} {m}：既有結果的設定不同（{bad}），會重跑")
                info = None
        if info is None:
            todo.append((t, m))
        else:
            results.append(info)
    # DOW 較短，放最後
    todo.sort(key=lambda j: j[0] == "DOW")
    print(f"共 {len(jobs)} 個工作：已完成 {len(results)}，待跑 {len(todo)}；workers={workers}")
    if todo:
        t0 = time.time()
        with ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context("spawn")) as ex:
            futs = {ex.submit(run_job, asdict(cfg), t, m): (t, m) for t, m in todo}
            for i, f in enumerate(as_completed(futs), 1):
                t, m = futs[f]
                try:
                    r = f.result()
                except Exception as e:  # noqa: BLE001
                    print(f"  [{i}/{len(todo)}] {t} {m} 失敗：{e!r}")
                    continue
                results.append(r)
                el = time.time() - t0
                extra = f" best={r['best_arch']}" if "best_arch" in r else ""
                print(f"  [{i}/{len(todo)}] {t:5s} {m:8s} MSE={r['MSE']:9.3f} R2={r['R2']:7.3f} "
                      f"({r['total_s']:.0f}s){extra} | 已用 {_fmt(el)}，預估剩 {_fmt(el / i * (len(todo) - i))}")
        print(f"完成，耗時 {_fmt(time.time() - t0)}")
    return pd.DataFrame(results)


# ---------------------------------------------------------------- 時間估算
def _bench_worker(cfg_dict: dict, ticker: str, n: int, seed: int):
    cfg = Config(**cfg_dict)
    data = prepare(load_prices(cfg, ticker), cfg)
    fobj = Fitness(cfg, data, os.devnull, seed)
    rng = np.random.default_rng(seed)
    times = []
    fobj(rng.uniform(0, 1, DIM))  # 第一次含 TF 初始化，不計
    for _ in range(n):
        t = time.time()
        fobj(rng.uniform(0, 1, DIM))
        times.append(time.time() - t)
    _tf(cfg).keras.backend.clear_session()
    model = build_baseline(cfg, "LSTM2D")
    t = time.time()
    if cfg.use_validation:
        fit_model(cfg, model, data.X_train, data.y_train, cfg.final_epochs, patience=cfg.final_patience, val_split=0.1)
    else:
        fit_model(cfg, model, data.X_train, data.y_train, cfg.final_epochs)
    return times, time.time() - t


def benchmark(cfg: Config, workers: int, n: int = 6, ticker: str = "AAPL"):
    """同時開 workers 個 process，各訓練 n 個隨機候選＋1 個最終 LSTM2D，量出負載下的有效時間。"""
    with ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context("spawn")) as ex:
        res = list(ex.map(_bench_worker, [asdict(cfg)] * workers, [ticker] * workers,
                          [n] * workers, range(workers)))
    t_search = float(np.mean([np.mean(r[0]) for r in res]))
    t_final = float(np.mean([r[1] for r in res]))
    return dict(search_s_per_model=t_search, final_s_per_model=t_final,
                search_eff=t_search / workers, final_eff=t_final / workers)


def project(cfg: Config, bench: dict, n_tickers: int, budget_h: float = 5.0) -> pd.DataFrame:
    """依 benchmark 推算每種方法的總時數，並建議 ARO 在預算內可用的最大迭代數。"""
    evals = cfg.pop_size * (cfg.iterations + 1)
    fin = bench["final_eff"] * cfg.final_seeds
    rows = [dict(model=m, hours=n_tickers * fin * c / 3600) for m, c in BASELINE_COST.items()]
    for m, u in UNIQUE_RATE.items():
        rows.append(dict(model=m, hours=n_tickers * (evals * u * bench["search_eff"] + fin) / 3600))
    df = pd.DataFrame(rows)
    per_ticker = budget_h * 3600 / n_tickers - fin
    max_iter = int(per_ticker / (cfg.pop_size * UNIQUE_RATE["LSTM-ARO"] * bench["search_eff"])) - 1
    df.attrs["aro_max_iterations"] = max_iter
    return df


# ---------------------------------------------------------------- 彙整
def collect(cfg: Config) -> pd.DataFrame:
    rows = []
    base = run_dir(cfg)
    if not os.path.isdir(base):
        return pd.DataFrame()
    for t in os.listdir(base):
        if not os.path.isdir(os.path.join(base, t)):
            continue
        for m in os.listdir(os.path.join(base, t)):
            if ".stale-" in m:
                continue
            if m.startswith(("LSTM-GA", "LSTM-ARO")) and not m.endswith(f"_p{cfg.pop_size}_i{cfg.iterations}"):
                continue  # 只取目前 pop/iters 的搜尋結果
            p = os.path.join(base, t, m, "done.json")
            if os.path.exists(p):
                with open(p, encoding="utf-8") as f:
                    rows.append(json.load(f))
    return pd.DataFrame(rows)


def metric_table(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    """對應論文 Table 7–10：列為股票、欄為模型。"""
    order = [m for m in ["LSTM-ARO", "LSTM-GA", "LSTM1D", "LSTM2D", "LSTM3D", "ANN", "NAIVE"] if m in set(df.model)]
    tab = df.pivot(index="ticker", columns="model", values=metric)[order]
    return tab.reindex([t for t in DJIA30 if t in tab.index])


def ranking(df: pd.DataFrame) -> pd.DataFrame:
    """每個指標：各模型拿第一的檔數與平均名次（論文的比較方式）。"""
    out = {}
    for metric in ["MSE", "MAE", "MAPE", "R2"]:
        tab = metric_table(df, metric)
        r = tab.rank(axis=1, ascending=metric != "R2")
        out[(metric, "wins")] = (r == 1).sum()
        out[(metric, "mean_rank")] = r.mean().round(2)
    return pd.DataFrame(out)


def wilcoxon_vs(df: pd.DataFrame, ref: str = "LSTM-ARO", metric: str = "MSE") -> pd.DataFrame:
    from scipy.stats import wilcoxon
    tab = metric_table(df, metric).dropna()
    rows = []
    for m in tab.columns:
        if m == ref:
            continue
        diff = tab[ref] - tab[m]
        p = wilcoxon(tab[ref], tab[m]).pvalue if (diff != 0).any() else 1.0
        rows.append(dict(vs=m, n=len(tab), ref_better=int((diff < 0).sum()), median_diff=float(diff.median()), p_value=float(p)))
    return pd.DataFrame(rows)


def time_table(df: pd.DataFrame) -> pd.DataFrame:
    """對應論文 Table 11：最終訓練時間（30 檔加總、每檔平均）；搜尋時間另列。"""
    g = df.groupby("model")
    out = pd.DataFrame({
        "final_train_total_s": g.final_s_per_seed.sum(),
        "final_train_per_stock_s": g.final_s_per_seed.mean(),
    })
    if "search_s" in df:
        out["search_total_h"] = g.search_s.sum() / 3600
        out["n_trained_mean"] = g.n_trained.mean()
    return out.round(2)


# ---------------------------------------------------------------- 協作：匯出與合併
def export_results(cfg: Config, tickers: List[str], zip_path: str) -> str:
    """把這些股票已完成的結果打包（相對於 run_dir：<TICKER>/<MODEL>/...）。不含 .stale 舊結果。"""
    import zipfile
    base = run_dir(cfg)
    n = 0
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for t in tickers:
            tdir = os.path.join(base, t)
            if not os.path.isdir(tdir):
                continue
            for m in os.listdir(tdir):
                mdir = os.path.join(tdir, m)
                if ".stale-" in m or not os.path.exists(os.path.join(mdir, "done.json")):
                    continue
                for fn in os.listdir(mdir):
                    z.write(os.path.join(mdir, fn), f"{t}/{m}/{fn}")
                n += 1
    print(f"已打包 {n} 個（股票 × 模型）→ {zip_path}")
    return zip_path


def merge_results(cfg: Config, zip_paths: List[str]) -> pd.DataFrame:
    """把協作者的 zip 合併進自己的 run_dir；設定或資料不一致的拒收，已存在的不覆蓋。"""
    import shutil
    import tempfile
    import zipfile
    base = run_dir(cfg)
    rows = []
    for zp in zip_paths:
        with tempfile.TemporaryDirectory() as tmp:
            with zipfile.ZipFile(zp) as z:
                z.extractall(tmp)
            for t in sorted(os.listdir(tmp)):
                for m in sorted(os.listdir(os.path.join(tmp, t))):
                    src = os.path.join(tmp, t, m)
                    dj = os.path.join(src, "done.json")
                    if not os.path.exists(dj):
                        continue
                    with open(dj, encoding="utf-8") as f:
                        info = json.load(f)
                    bad = compatible(cfg, info)
                    dst = os.path.join(base, t, m)
                    if bad:
                        status = f"拒收：設定不同 {bad}"
                    elif os.path.exists(os.path.join(dst, "done.json")):
                        status = "已存在，略過"
                    else:
                        if os.path.exists(dst):
                            shutil.rmtree(dst)  # 自己未完成的半成品
                        shutil.copytree(src, dst)
                        status = "合併"
                    rows.append(dict(zip=os.path.basename(zp), ticker=t, model=m, status=status))
    df = pd.DataFrame(rows)
    if len(df):
        print(df.status.value_counts().to_string())
    return df


def progress(cfg: Config, tickers: List[str]) -> pd.DataFrame:
    """每檔股票 × 模型的完成狀態：✓ 完成、… 搜尋中（有候選紀錄）、空白 = 未開始。"""
    tab = pd.DataFrame("", index=tickers, columns=ALL_MODELS)
    for t in tickers:
        for m in ALL_MODELS:
            d = model_dir(cfg, t, m)
            if os.path.exists(os.path.join(d, "done.json")):
                tab.loc[t, m] = "✓"
            elif os.path.exists(os.path.join(d, "candidates.jsonl")):
                with open(os.path.join(d, "candidates.jsonl"), encoding="utf-8") as f:
                    tab.loc[t, m] = f"…{sum(1 for _ in f)}"
    return tab

In [ ]:
import djia_lib as L
importlib.reload(L)
import pandas as pd

DATA_CSV = os.path.join(BASE, "data", "djia30_2018_2022.csv")

def make_cfg(use_gpu=False, data_hash=""):
    return L.Config(
        data_csv=DATA_CSV, out_dir=os.path.join(BASE, "results_djia"),
        price_col=PRICE_COL, use_validation=USE_VALIDATION,
        pop_size=POP_SIZE, iterations=ITERATIONS,
        search_epochs=SEARCH_EPOCHS, final_epochs=FINAL_EPOCHS, batch_size=BATCH_SIZE,
        final_seeds=FINAL_SEEDS, seed=SEED, threads=THREADS, use_gpu=bool(use_gpu), data_hash=data_hash,
    )

## 4. 資料（所有人用同一份凍結檔）

- 優先從 `DATA_URL`（GitHub 上負責人凍結的 CSV）下載；拿不到才改從 Yahoo Finance 下載，此時資料指紋會和別人不同，**結果不能合併**。
- 範圍 2018-01-01 ～ 2023-01-01。**WBA** 已下市（29 檔）；**DOW** 2019-03-20 才上市；**WMT** 2024 年 1 拆 3，與論文比 MSE 要 ×9。

In [ ]:
if not os.path.exists(DATA_CSV):
    os.makedirs(os.path.dirname(DATA_CSV), exist_ok=True)
    try:
        import urllib.request
        urllib.request.urlretrieve(DATA_URL, DATA_CSV)
        print("已從 DATA_URL 下載凍結資料")
    except Exception as e:
        print(f"⚠ 無法從 DATA_URL 下載（{e}）；改用 yfinance。這份資料的指紋會和其他人不同，結果不能合併！")
        extra = os.path.join(BASE, "data", "extra")
        L.download_djia(make_cfg(), extra_dir=extra if os.path.isdir(extra) else None)

DATA_HASH = L.file_hash(DATA_CSV)
cfg = make_cfg(data_hash=DATA_HASH)
ALL_TICKERS = L.available_tickers(cfg)
TICKER_LIST = L.shard_tickers(ALL_TICKERS, SHARD)
print(f"RUN ID：{L.run_id(cfg)}    ← 所有人必須相同")
print(f"資料共 {len(ALL_TICKERS)} 檔；本次負責（SHARD={SHARD}）{len(TICKER_LIST)} 檔：", " ".join(TICKER_LIST))
display(L.progress(cfg, TICKER_LIST))

## 5. 時間估算

同時開 `WORKERS` 個 process，各訓練幾個隨機候選與一個 LSTM2D，量出負載下的速度，推算**本次負責的股票**要跑多久。
有 GPU 時 CPU / GPU 各量一次，選較快的。結果存在 Drive，斷線重跑時直接沿用（`REBENCH = True` 才重量）。

In [ ]:
bench_path = os.path.join(BASE, f"bench_{'colab' if IN_COLAB else 'local'}.json")
has_gpu = IN_COLAB and os.system("nvidia-smi > /dev/null 2>&1") == 0
if os.path.exists(bench_path) and not REBENCH:
    saved = json.load(open(bench_path))
    bench = {bool(int(k)): v for k, v in saved["bench"].items()}
    print("沿用上次的時間估算", bench_path)
else:
    devices = [USE_GPU] if USE_GPU is not None else ([False, True] if has_gpu else [False])
    tick = "AAPL" if "AAPL" in ALL_TICKERS else ALL_TICKERS[0]
    bench = {dev: L.benchmark(make_cfg(dev, DATA_HASH), WORKERS, n=4, ticker=tick) for dev in devices}
    json.dump({"bench": {str(int(k)): v for k, v in bench.items()}, "workers": WORKERS}, open(bench_path, "w"))
for dev, b in bench.items():
    print(f"{'GPU' if dev else 'CPU'}：搜尋候選 {b['search_s_per_model']:.1f}s/個、最終模型 {b['final_s_per_model']:.1f}s/個"
          f"（{WORKERS} workers 同時跑）")
best_dev = USE_GPU if USE_GPU is not None else min(bench, key=lambda d: bench[d]["search_eff"])
if best_dev not in bench:
    best_dev = min(bench, key=lambda d: bench[d]["search_eff"])
cfg = make_cfg(best_dev, DATA_HASH)
print("使用", "GPU" if best_dev else "CPU", "" if best_dev or not has_gpu else "→ 可改用 CPU 執行階段，省下 GPU 額度")

proj = L.project(cfg, bench[best_dev], len(TICKER_LIST), TIME_BUDGET_H).set_index("model")
display(proj.round(2).rename(columns={"hours": f"預估時數（{len(TICKER_LIST)} 檔）"}))
total = proj.hours.sum()
print(f"本次合計約 {total:.1f} 小時。", "⚠ 超過 10 小時，免費版 Colab 很可能中途斷線（可續跑，但建議請負責人切更多份）" if IN_COLAB and total > 10 else "")

## Step 1：NAIVE、ANN、LSTM1D、LSTM2D、LSTM3D（固定架構，論文 Table 1–4）

In [ ]:
res_base = L.run_jobs(cfg, ["NAIVE", "ANN", "LSTM1D", "LSTM2D", "LSTM3D"], TICKER_LIST, WORKERS)

## Step 2：LSTM-GA

族群與迭代數與 ARO 相同、初始族群相同；tournament 3、uniform crossover 0.8、mutation 1/9、菁英 1（論文未給 GA 細節）。

In [ ]:
res_ga = L.run_jobs(cfg, ["LSTM-GA"], TICKER_LIST, WORKERS)

## Step 3：LSTM-ARO

In [ ]:
res_aro = L.run_jobs(cfg, ["LSTM-ARO"], TICKER_LIST, WORKERS)

## 匯出結果（協作者：把產生的 zip 傳給負責人）

In [ ]:
os.makedirs(os.path.join(BASE, "exports"), exist_ok=True)
tag = f"shard{SHARD.replace('/', 'of')}" if SHARD else "all"
zip_path = os.path.join(BASE, "exports", f"results_{L.run_id(cfg)}_{tag}.zip")
L.export_results(cfg, TICKER_LIST, zip_path)
display(L.progress(cfg, TICKER_LIST))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## 合併協作者的結果（負責人）

把收到的 zip 放進 Drive 的 `lstm_aro_djia/incoming/` 再執行；已合併過的會自動略過。

In [ ]:
incoming = sorted(glob.glob(os.path.join(BASE, "incoming", "*.zip")))
if incoming:
    merged = L.merge_results(cfg, incoming)
    display(merged[merged.status != "合併"]) if len(merged) else None
else:
    print("incoming/ 沒有 zip，略過")
print("全部股票的完成狀態（✓ 完成、…N 搜尋中已訓練 N 個候選）")
display(L.progress(cfg, ALL_TICKERS))

## 彙整：對應論文 Table 7–11

- 指標都在原價格尺度，MAPE 為比例；每格是 `FINAL_SEEDS` 個 seed 的平均。
- 與論文比較時看**名次與方向**，不要直接比絕對值（資料版本、未公開的訓練細節都不同）。

In [ ]:
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
allres = L.collect(cfg)
print(f"已完成 {len(allres)} 個（股票 × 模型）")
if len(allres):
    tables = {}
    for i, metric in enumerate(["MSE", "MAE", "MAPE", "R2"], start=7):
        tables[metric] = L.metric_table(allres, metric)
        print(f"\nTable {i}  {metric}")
        display(tables[metric])
    print("\n各指標拿第一的檔數與平均名次")
    display(L.ranking(allres))
    if {"LSTM-ARO"} <= set(allres.model):
        print("\nLSTM-ARO 與其他模型的 MSE 配對檢定（Wilcoxon signed-rank，逐檔配對）")
        try:
            display(L.wilcoxon_vs(allres))
        except Exception as e:
            print("檢定略過：", e)
    print("\nTable 11 對應：時間")
    display(L.time_table(allres))

    summ = os.path.join(L.run_dir(cfg), f"summary_p{cfg.pop_size}_i{cfg.iterations}")
    os.makedirs(summ, exist_ok=True)
    for metric, tab in tables.items():
        tab.to_csv(os.path.join(summ, f"table_{metric}.csv"))
    L.ranking(allres).to_csv(os.path.join(summ, "ranking.csv"))
    L.time_table(allres).to_csv(os.path.join(summ, "time.csv"))
    allres.drop(columns=["config"], errors="ignore").to_csv(os.path.join(summ, "all_results.csv"), index=False)
    print("已存到", summ)

In [ ]:
# 搜尋收斂與最佳架構
import matplotlib.pyplot as plt
srch = allres[allres.model.isin(L.SEARCHERS)] if len(allres) else allres
if len(srch):
    show = [t for t in ALL_TICKERS if t in set(srch.ticker)][:6]
    fig, axes = plt.subplots(2, 3, figsize=(14, 7)); axes = axes.ravel()
    for ax, t in zip(axes, show):
        for _, r in srch[srch.ticker == t].iterrows():
            ax.plot(r["history"], marker="o", ms=3, label=r["model"])
        ax.set_title(t); ax.set_xlabel("iteration"); ax.set_ylabel("best fitness (MSE)"); ax.legend()
    plt.tight_layout(); plt.show()
    display(srch[["ticker", "model", "best_arch", "best_fitness", "n_trained", "search_s"]].sort_values(["ticker", "model"]))

In [ ]:
# 單檔預測圖（各模型 seed 平均的測試集預測）
TICK = TICKER_LIST[0]
fig, ax = plt.subplots(figsize=(13, 5))
first = True
for m in ["NAIVE", "ANN", "LSTM1D", "LSTM2D", "LSTM3D", "LSTM-GA", "LSTM-ARO"]:
    p = os.path.join(L.model_dir(cfg, TICK, m), "predictions.csv")
    if not os.path.exists(p):
        continue
    d = pd.read_csv(p, parse_dates=["date"])
    if first:
        ax.plot(d.date, d.actual, color="black", lw=2, label="actual"); first = False
    ax.plot(d.date, d.pred, lw=1, label=m)
ax.set_title(f"{TICK} test set"); ax.legend(ncol=4); plt.show()